# Catalog Exploration — Unity Catalog vs Legacy Hive Metastore
**Day 4 | Run this notebook on `dev-cluster` to understand exactly what you see in the Catalog UI**

---

## What you see in the Catalog tab (left sidebar)

```
Catalog tab
├── My organization                  ← your Unity Catalog Metastore (the invisible wrapper)
│     ├── dbw_ev_intelligence_dev    ← YOUR catalog (UC) — use this for everything
│     └── system                     ← Databricks internal catalog (read-only, ignore)
│
├── Shares received
│     └── samples                    ← Delta Sharing demo data (read-only, ignore)
│
└── Legacy
      └── hive_metastore             ← OLD workspace Hive metastore (pre-UC)
            └── default              ← schema inside old metastore (DO NOT use)
```

**The metastore itself** is not a clickable node — it is the account-level container that holds everything
under "My organization". You see its catalogs, not the metastore object itself.

---

## What is `hive_metastore` (Legacy)?

When your Databricks workspace was first created, it auto-created a workspace-level Hive metastore.
This is the old way — before Unity Catalog existed. When UC was enabled, Databricks moved it under
**Legacy** instead of deleting it so existing tables would not break.

| | hive_metastore (Legacy) | dbw_ev_intelligence_dev (UC) |
|---|---|---|
| Namespace | `schema.table` (2-level) | `catalog.schema.table` (3-level) |
| Visible across workspaces | NO — this workspace only | YES — all workspaces in same region |
| Column-level security | No | Yes |
| Data lineage | No | Yes |
| External Locations / Volumes | No | Yes |
| Permissions tab in UI | No | Yes |

**Rule: Never create tables under `hive_metastore` for this project. Always use `dbw_ev_intelligence_dev`.**

---

## How to run this notebook

Run each cell top to bottom. Every cell prints its own explanation before showing results.
No manual edits needed — all values are already set to our project's names.

In [ ]:
# =============================================================
# CELL 1 — Check which catalog Databricks defaults to
# =============================================================
# RISK: If you run SQL without specifying a catalog, Databricks
# may silently write to hive_metastore (Legacy) instead of UC.
# This cell tells you what the current default is.
# =============================================================

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
current_schema  = spark.sql("SELECT current_schema()").collect()[0][0]

print(f"Current catalog : {current_catalog}")
print(f"Current schema  : {current_schema}")
print()

if current_catalog == "hive_metastore":
    print("WARNING: Default is hive_metastore (Legacy).")
    print("Run the next cell to switch to Unity Catalog before doing anything else.")
elif current_catalog == "dbw_ev_intelligence_dev":
    print("OK: Already on Unity Catalog.")
else:
    print(f"INFO: Current catalog is '{current_catalog}'.")
    print("Run the next cell to set it explicitly to Unity Catalog.")

In [ ]:
# =============================================================
# CELL 2 — Set catalog and schema to Unity Catalog
# =============================================================
# Run this at the top of EVERY notebook before any SQL or
# table operations. This ensures all subsequent SELECT,
# CREATE TABLE, INSERT go to UC — not Legacy hive_metastore.
# =============================================================

UC_CATALOG = "dbw_ev_intelligence_dev"
UC_SCHEMA  = "default"

spark.sql(f"USE CATALOG {UC_CATALOG}")
spark.sql(f"USE SCHEMA {UC_SCHEMA}")

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
current_schema  = spark.sql("SELECT current_schema()").collect()[0][0]

print(f"Active catalog : {current_catalog}")
print(f"Active schema  : {current_schema}")
print()
print("All SQL in this notebook now targets Unity Catalog.")
print(f"Short table names resolve to: {current_catalog}.{current_schema}.<table_name>")

In [ ]:
# =============================================================
# CELL 3 — List ALL catalogs visible to this workspace
# =============================================================
# This is everything you see in the Catalog UI tree:
#   My organization  → your UC catalogs
#   Shares received  → Delta Sharing catalogs (read-only)
#   Legacy           → old hive_metastore
# =============================================================

print("=" * 60)
print("ALL CATALOGS VISIBLE TO THIS WORKSPACE")
print("=" * 60)

catalogs = spark.sql("SHOW CATALOGS").collect()

for row in catalogs:
    name = row[0]
    if name == "hive_metastore":
        label = "← Legacy (pre-UC) — DO NOT use for new work"
    elif name == "system":
        label = "← Databricks internal catalog (read-only)"
    elif name == "samples":
        label = "← Delta Sharing demo catalog (read-only)"
    elif name == "dbw_ev_intelligence_dev":
        label = "← YOUR Unity Catalog — use this for everything"
    else:
        label = ""
    print(f"  {name:35s} {label}")

In [ ]:
# =============================================================
# CELL 4 — List schemas inside UC vs Legacy side by side
# =============================================================
# Schema = second level of the namespace tree.
# In the Catalog UI: expand a catalog → you see schemas.
# UC schemas are shared across workspaces.
# hive_metastore schemas exist only in this workspace.
# =============================================================

print("=" * 60)
print("SCHEMAS INSIDE UNITY CATALOG: dbw_ev_intelligence_dev")
print("(these are shared across ALL workspaces in the same region)")
print("=" * 60)
uc_schemas = spark.sql("SHOW SCHEMAS IN dbw_ev_intelligence_dev").collect()
for row in uc_schemas:
    print(f"  dbw_ev_intelligence_dev.{row[0]}")

print()
print("=" * 60)
print("SCHEMAS INSIDE LEGACY: hive_metastore")
print("(these are visible in THIS workspace only — do not use)")
print("=" * 60)
hive_schemas = spark.sql("SHOW SCHEMAS IN hive_metastore").collect()
for row in hive_schemas:
    print(f"  hive_metastore.{row[0]}")

In [ ]:
# =============================================================
# CELL 5 — List tables and volumes inside your UC schema
# =============================================================
# Third level of the namespace — the actual objects you work with.
# Tables = queryable via SQL
# Volumes = file paths accessible via /Volumes/... in notebooks
# =============================================================

print("=" * 60)
print("TABLES in dbw_ev_intelligence_dev.default")
print("=" * 60)
tables = spark.sql("SHOW TABLES IN dbw_ev_intelligence_dev.default").collect()
if tables:
    for row in tables:
        print(f"  dbw_ev_intelligence_dev.default.{row['tableName']}")
else:
    print("  (no tables yet — will be created in Silver layer, Day 5+)")

print()
print("=" * 60)
print("VOLUMES in dbw_ev_intelligence_dev.default")
print("=" * 60)
volumes = spark.sql("SHOW VOLUMES IN dbw_ev_intelligence_dev.default").collect()
if volumes:
    for row in volumes:
        print(f"  /Volumes/dbw_ev_intelligence_dev/default/{row['volume_name']}")
else:
    print("  (no volumes yet)")

In [ ]:
# =============================================================
# CELL 6 — hive_metastore vs UC: full comparison printed
# =============================================================
# hive_metastore is workspace-scoped — tables there are invisible
# to any other workspace. Unity Catalog tables are account-scoped
# — visible to all workspaces attached to the same metastore.
# =============================================================

print("=" * 60)
print("SCOPE COMPARISON: hive_metastore vs Unity Catalog")
print("=" * 60)
print()
print("hive_metastore (Legacy):")
print("  Visible in     : THIS workspace only")
print("  Namespace      : schema.table  (2-level)")
print("  Governance     : none — no column/row security, no lineage")
print("  External Locs  : not supported")
print("  Volumes        : not supported")
print("  Status         : deprecated — Databricks is moving away from this")
print()
print("dbw_ev_intelligence_dev (Unity Catalog):")
print("  Visible in     : ALL workspaces attached to the same metastore")
print("  Namespace      : catalog.schema.table  (3-level)")
print("  Governance     : column-level, row-level, lineage, permissions UI")
print("  External Locs  : yes — evdatalakedev-bronze / silver / gold")
print("  Volumes        : yes — /Volumes/dbw_ev_intelligence_dev/default/bronze-volume")
print("  Status         : current standard — use for all new work")
print()
print("=" * 60)
print("VERDICT: Always use dbw_ev_intelligence_dev for this project.")
print("=" * 60)

In [ ]:
# =============================================================
# CELL 7 — Browse the Bronze Volume path
# =============================================================
# /Volumes/... is the UC-governed path to your ADLS Bronze container.
# No ADLS credentials needed in this code — UC handles auth via
# the Storage Credential (cred-ev-intelligence-dev).
# If this fails, see the error message for the exact fix.
# =============================================================

BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"

print(f"Listing root of Bronze Volume: {BRONZE_VOLUME}")
print()

try:
    items = dbutils.fs.ls(BRONZE_VOLUME)
    for item in items:
        kind = "DIR " if item.isDir() else "FILE"
        print(f"  {kind}  {item.path}")
    print()
    print("Volume accessible — UC External Location + Storage Credential working correctly.")
except Exception as e:
    print(f"ERROR: {e}")
    print()
    print("Possible causes:")
    print("  1. Volume 'bronze-volume' does not exist — create it via Catalog UI")
    print("  2. Your user does not have READ VOLUME privilege — run:")
    print("     GRANT READ VOLUME ON VOLUME dbw_ev_intelligence_dev.default.`bronze-volume`")
    print("     TO `your.email@company.com`;")
    print("  3. Cluster access mode is not Single User or Shared — check Compute settings")

In [ ]:
# =============================================================
# CELL 8 — Full namespace reference card for this project
# =============================================================
# Quick-reference for every path format used in the EV project.
# Bookmark this cell — when you are unsure which path to use,
# this is the single source of truth.
# =============================================================

print("=" * 65)
print("NAMESPACE REFERENCE CARD — EV Intelligence Project")
print("=" * 65)
print()
print("UNITY CATALOG (use these in all new notebooks):")
print("  Metastore    : dbw_ev_intelligence_dev  (account-level, never in code)")
print("  Catalog      : dbw_ev_intelligence_dev")
print("  Schema       : default  (bronze/silver/gold schemas planned for later)")
print("  Table (SQL)  : dbw_ev_intelligence_dev.default.<table_name>")
print("  Volume (code): /Volumes/dbw_ev_intelligence_dev/default/bronze-volume/")
print()
print("ADLS PATHS (used by ADF and registered as UC External Locations):")
print("  Bronze : abfss://bronze@evdatalakedev.dfs.core.windows.net/")
print("  Silver : abfss://silver@evdatalakedev.dfs.core.windows.net/")
print("  Gold   : abfss://gold@evdatalakedev.dfs.core.windows.net/")
print()
print("SOURCE BLOB (read-only — instructor-provided):")
print("  wasbs://source@dataenggdailystorage.blob.core.windows.net/")
print("  realtime/charging_sessions/YYYY/MM/DD/HH/*.csv")
print()
print("LEGACY (do NOT use for new work):")
print("  hive_metastore.default.<table>  ← workspace-only, no governance")
print()
print("=" * 65)